In [1]:
from pyspark.sql import SparkSession

In [4]:
spark = SparkSession.builder.master("local[*]").appName("test").getOrCreate()

In [5]:
spark

In [7]:
from google.colab import files
uploaded = files.upload()

Saving customers.txt.txt to customers.txt (1).txt


In [12]:
orders_base = spark.sparkContext.textFile("orders.txt")

In [13]:
orders_mapped = orders_base.map(lambda x:(x.split(",")[2],x.split(",")[3]))

In [41]:
orders_mapped.sortByKey().take(5)

[('1', 'COMPLETE'),
 ('10', 'COMPLETE'),
 ('10', 'COMPLETE'),
 ('100', 'COMPLETE'),
 ('100', 'PROCESSING')]

In [15]:
# customers.txt (1).txt
customers_base = spark.sparkContext.textFile("customers.txt (1).txt")

In [23]:
customers_base.take(5)

['1,Richard,Hernandez,6303 Heather Plaza,Brownsville,TX,78521',
 '2,Mary,Barrett,9526 Noble Embers Ridge,Littleton,CO,80126',
 '3,Ann,Smith,3422 Blue Pioneer Bend,Caguas,PR,00725',
 '4,Mary,Jones,8324 Little Common,San Marcos,CA,92069',
 '5,Robert,Hudson,10 Crystal River Mall ,Caguas,PR,00725']

In [34]:
customers_mapped = customers_base.map(lambda x: (x.split(",")[0],x.split(",")[6]))

In [35]:
customers_mapped.take(5)

[('1', '78521'),
 ('2', '80126'),
 ('3', '00725'),
 ('4', '92069'),
 ('5', '00725')]

In [36]:
joined_rdd = customers_mapped.join(orders_mapped)

In [40]:
joined_rdd.sortByKey().take(20)

[('1', ('78521', 'COMPLETE')),
 ('10', ('22554', 'COMPLETE')),
 ('10', ('22554', 'COMPLETE')),
 ('100', ('00725', 'COMPLETE')),
 ('100', ('00725', 'PROCESSING')),
 ('100', ('00725', 'CANCELED')),
 ('100', ('00725', 'COMPLETE')),
 ('100', ('00725', 'COMPLETE')),
 ('100', ('00725', 'PENDING_PAYMENT')),
 ('100', ('00725', 'PENDING')),
 ('1000', ('11520', 'COMPLETE')),
 ('1000', ('11520', 'PROCESSING')),
 ('1000', ('11520', 'PENDING')),
 ('1000', ('11520', 'COMPLETE')),
 ('1000', ('11520', 'COMPLETE')),
 ('1000', ('11520', 'CLOSED')),
 ('1000', ('11520', 'COMPLETE')),
 ('10000', ('00725', 'COMPLETE')),
 ('10000', ('00725', 'PENDING_PAYMENT')),
 ('10000', ('00725', 'COMPLETE'))]

In [42]:
#broadcasting the smaller dataset
customers_broadcast = spark.sparkContext.broadcast(customers_mapped.collect())

In [43]:
def get_pincode(customer_id):
  try:
    return customers_broadcast.value[customer_id]
  except:
    return "-1"

In [44]:
joined_rdd = orders_mapped.map(lambda x: (get_pincode(int(x[0])),x[1]))

In [51]:
joined_rdd.take(5)

[(('11600', '00725'), 'CLOSED'),
 (('257', '00791'), 'PENDING_PAYMENT'),
 (('12112', '00725'), 'COMPLETE'),
 (('8828', '00725'), 'CLOSED'),
 (('11319', '00612'), 'COMPLETE')]

In [54]:
spark.stop()